# 11 - Image : Modélisation Avancée (CLIP + Fine-Tuning CNN)

## Objectif de ce notebook
Explorer des approches **avancées** pour la classification d'images :
1. **CLIP ViT-L/14** : Features vision-langage (LAION-2B) + classifieurs ML optimisés
2. **Optimisation classifieurs** : GridSearch SVM / LR + MLP sur features CLIP
3. **Fine-Tuning EfficientNet-B0** : Adaptation d'un CNN pré-entraîné ImageNet

Comparaison avec la baseline ResNet50 du NB09.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import pickle
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

import open_clip

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
NUM_THREADS = os.cpu_count() or 4
torch.set_num_threads(NUM_THREADS)

In [ ]:
ROOT = Path('../').resolve()
DATA_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
IMAGE_CLEAN_DIR = DATA_DIR / 'image_clean'

with open(MODELS_DIR / 'image_df.pkl', 'rb') as f:
    image_df = pickle.load(f)

# Encoder les labels
with open(MODELS_DIR / 'label_encoder_image.pkl', 'rb') as f:
    le = pickle.load(f)

image_df['label'] = le.transform(image_df['prdtypecode'])
NUM_CLASSES = len(le.classes_)

# Split train/val (même seed que NB09)
train_df, val_df = train_test_split(image_df, test_size=0.2, random_state=SEED, stratify=image_df['label'])
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Classes: {NUM_CLASSES}')

## 2. CLIP ViT-L/14 : Extraction de features + Classification optimisée

Utilisation du modèle **CLIP ViT-L/14** (OpenCLIP, pré-entraîné sur LAION-2B) pour extraire des features visuelles riches (768-dim vs 512-dim pour ViT-B/32), puis classification avec des modèles ML optimisés via GridSearch.

**Améliorations vs version précédente (ViT-B/32)** :
- Modèle CLIP plus puissant (ViT-L/14 : 304M params vs ViT-B/32 : 88M params)
- Features 768-dim (vs 512-dim), plus discriminantes
- Optimisation des hyperparamètres (GridSearchCV)
- Ajout d'un classifieur MLP (réseau de neurones)

In [ ]:
CLIP_MODEL_NAME = 'ViT-L-14'
CLIP_PRETRAINED = 'laion2b_s32b_b82k'
CLIP_CACHE_L14 = MODELS_DIR / 'clip_features_cache_vit_l14.npz'

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
)
clip_model = clip_model.to(DEVICE).eval()
print(f'CLIP model: {CLIP_MODEL_NAME} ({CLIP_PRETRAINED})')

def extract_clip_features(df, batch_size=32):
    all_features = []
    valid_idx = []
    paths = df['image_path_clean'].tolist()
    total = len(paths)
    
    for start in range(0, total, batch_size):
        batch_paths = paths[start:start + batch_size]
        images = []
        batch_valid = []
        for j, p in enumerate(batch_paths):
            try:
                img = Image.open(p).convert('RGB')
                images.append(clip_preprocess(img))
                batch_valid.append(start + j)
            except Exception:
                continue
        if not images:
            continue
        batch_tensor = torch.stack(images).to(DEVICE)
        with torch.no_grad():
            feats = clip_model.encode_image(batch_tensor)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_features.append(feats.cpu().numpy())
        valid_idx.extend(batch_valid)
        if (start // batch_size) % 50 == 0:
            print(f'  {min(start + batch_size, total)}/{total}')
    
    return np.vstack(all_features), valid_idx

if CLIP_CACHE_L14.exists():
    print('Chargement features CLIP ViT-L/14 (cache)...')
    cache = np.load(CLIP_CACHE_L14)
    X_clip_train = cache['X_train']
    X_clip_val = cache['X_val']
    y_train_clip = cache['y_train']
    y_val_clip = cache['y_val']
else:
    print('Extraction features CLIP ViT-L/14 (train)...')
    X_clip_train, idx_train = extract_clip_features(train_df)
    y_train_clip = train_df.iloc[idx_train]['label'].values
    
    print('Extraction features CLIP ViT-L/14 (val)...')
    X_clip_val, idx_val = extract_clip_features(val_df)
    y_val_clip = val_df.iloc[idx_val]['label'].values
    
    np.savez(CLIP_CACHE_L14, X_train=X_clip_train, X_val=X_clip_val,
             y_train=y_train_clip, y_val=y_val_clip)
    print(f'Cache: {CLIP_CACHE_L14}')

print(f'Features CLIP ViT-L/14 : train={X_clip_train.shape}, val={X_clip_val.shape}')

### Classification sur features CLIP ViT-L/14 (GridSearch + MLP)

In [ ]:
clip_results = {}

# --- 1. Logistic Regression avec GridSearch ---
print('=' * 60)
print('1. Logistic Regression (GridSearchCV)')
print('=' * 60)
lr_param_grid = {'C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED, solver='lbfgs'),
    lr_param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
lr_grid.fit(X_clip_train, y_train_clip)
print(f'  Meilleur C : {lr_grid.best_params_["C"]}')
print(f'  F1-macro CV : {lr_grid.best_score_:.4f}')

y_pred_lr = lr_grid.predict(X_clip_val)
f1_lr = f1_score(y_val_clip, y_pred_lr, average='macro')
print(f'  F1-macro Val : {f1_lr:.4f}')
clip_results['CLIP ViT-L/14 + LR (opt.)'] = {'f1': f1_lr, 'pred': y_pred_lr}

# --- 2. LinearSVC avec GridSearch ---
print('\n' + '=' * 60)
print('2. LinearSVC (GridSearchCV)')
print('=' * 60)
svm_param_grid = {'C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
svm_grid = GridSearchCV(
    LinearSVC(class_weight='balanced', random_state=SEED, dual=False, max_iter=3000),
    svm_param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
svm_grid.fit(X_clip_train, y_train_clip)
print(f'  Meilleur C : {svm_grid.best_params_["C"]}')
print(f'  F1-macro CV : {svm_grid.best_score_:.4f}')

y_pred_svm = svm_grid.predict(X_clip_val)
f1_svm = f1_score(y_val_clip, y_pred_svm, average='macro')
print(f'  F1-macro Val : {f1_svm:.4f}')
clip_results['CLIP ViT-L/14 + SVM (opt.)'] = {'f1': f1_svm, 'pred': y_pred_svm}

# --- 3. MLP Classifier ---
print('\n' + '=' * 60)
print('3. MLP Classifier')
print('=' * 60)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_clip_train)
X_val_scaled = scaler.transform(X_clip_val)

mlp_clip = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    batch_size=256,
    learning_rate='adaptive',
    learning_rate_init=1e-3,
    random_state=SEED,
    verbose=True
)
mlp_clip.fit(X_train_scaled, y_train_clip)
y_pred_mlp = mlp_clip.predict(X_val_scaled)
f1_mlp = f1_score(y_val_clip, y_pred_mlp, average='macro')
print(f'  F1-macro Val : {f1_mlp:.4f}')
clip_results['CLIP ViT-L/14 + MLP'] = {'f1': f1_mlp, 'pred': y_pred_mlp}

# --- Recap CLIP ---
print('\n' + '=' * 60)
print('RECAP CLIP ViT-L/14')
print('=' * 60)
best_clip_name = max(clip_results, key=lambda k: clip_results[k]['f1'])
for name, res in sorted(clip_results.items(), key=lambda x: -x[1]['f1']):
    marker = ' <-- best' if name == best_clip_name else ''
    print(f"  {name:<40} F1-macro = {res['f1']:.4f}{marker}")

best_clip_f1 = clip_results[best_clip_name]['f1']
best_clip_pred = clip_results[best_clip_name]['pred']

print(f'\nClassification Report ({best_clip_name}) :')
print(classification_report(y_val_clip, best_clip_pred, target_names=[str(c) for c in le.classes_]))

## 3. Fine-Tuning EfficientNet-B0

Adaptation du CNN **EfficientNet-B0** pré-entraîné ImageNet :
- Gel de toutes les couches sauf les 2 derniers blocs + classificateur
- Class weights dans la loss `CrossEntropyLoss`
- Learning rates différentiels (backbone vs classificateur)

In [ ]:
class ProductImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df['image_path_clean'].tolist()
        self.labels = df['label'].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = ProductImageDataset(train_df, train_transform)
val_ds = ProductImageDataset(val_df, val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

# Class weights pour la loss
cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_df['label'].values)
cw_tensor = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw_tensor)

# EfficientNet-B0
eff_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Geler tout
for param in eff_model.parameters():
    param.requires_grad = False

# D\u00e9geler les 2 derniers blocs (features[6] et features[7])
for param in eff_model.features[6].parameters():
    param.requires_grad = True
for param in eff_model.features[7].parameters():
    param.requires_grad = True

# Remplacer le classificateur
in_features = eff_model.classifier[1].in_features
eff_model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, NUM_CLASSES)
)
eff_model = eff_model.to(DEVICE)

trainable = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in eff_model.parameters())
print(f'Param\u00e8tres : {trainable:,} entra\u00eenables / {total:,} total ({100*trainable/total:.1f}%)')

In [ ]:
NUM_EPOCHS = 10

backbone_params = list(eff_model.features[6].parameters()) + list(eff_model.features[7].parameters())
classifier_params = list(eff_model.classifier.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-4},
    {'params': classifier_params, 'lr': 1e-3}
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    eff_model.train()
    running_loss = 0.0
    n_batches = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = eff_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    scheduler.step()
    train_loss = running_loss / n_batches

    eff_model.eval()
    val_loss_total = 0.0
    val_batches = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = eff_model(images)
            val_loss_total += criterion(outputs, labels).item()
            val_batches += 1
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = val_loss_total / val_batches
    val_f1 = f1_score(all_labels, all_preds, average='macro')

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)

    marker = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(eff_model.state_dict(), MODELS_DIR / 'efficientnet_best.pth')
        marker = ' <-- best'

    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1-macro: {val_f1:.4f}{marker}')

print(f'\nMeilleur F1-macro EfficientNet : {best_f1:.4f}')

## 4. Comparaison et Conclusions

In [ ]:
# Courbes d'entrainement
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)
ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss par epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['val_f1'], marker='o', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1-macro')
ax2.set_title('F1-macro (validation) par epoch')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'efficientnet_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Charger la baseline du NB09
baseline_csv = MODELS_DIR / 'image_baseline_comparison.csv'
if baseline_csv.exists():
    baseline_df = pd.read_csv(baseline_csv)
    best_baseline = baseline_df[baseline_df['Weights'] == 'Oui'].sort_values('F1_macro', ascending=False).iloc[0]
    baseline_f1 = best_baseline['F1_macro']
    baseline_name = best_baseline['Model']
else:
    baseline_f1 = 0.0
    baseline_name = 'N/A'

# Tableau comparatif complet
comparison_rows = [
    {'Model': f'ResNet50 baseline ({baseline_name})', 'F1_macro': baseline_f1},
    {'Model': 'EfficientNet-B0 (fine-tuning)', 'F1_macro': best_f1},
]
for name, res in clip_results.items():
    comparison_rows.append({'Model': name, 'F1_macro': res['f1']})

comparison = pd.DataFrame(comparison_rows).sort_values('F1_macro', ascending=False)

print('\nCOMPARAISON FINALE - TOUS LES MODELES IMAGE')
print('=' * 70)
for _, row in comparison.iterrows():
    print(f"  {row['Model']:<45} F1-macro = {row['F1_macro']:.4f}")

# Graphique comparatif
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(comparison))]
bars = ax.barh(range(len(comparison)), comparison['F1_macro'].values, color=colors)
ax.set_yticks(range(len(comparison)))
ax.set_yticklabels(comparison['Model'].values)
ax.set_xlabel('F1-macro')
ax.set_title('Comparaison des modeles image (validation)')
ax.set_xlim(0, 1)
for bar, val in zip(bars, comparison['F1_macro'].values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(MODELS_DIR / 'image_all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

comparison.to_csv(MODELS_DIR / 'image_advanced_comparison.csv', index=False)
print(f'\nCSV: models/image_advanced_comparison.csv')